In [1]:
# if you are using Jupyter notebook execute 1st:
!pip install tensorflow

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Dense
# Dense: to add layers(hidden, output)
from tensorflow.keras import Sequential
# Sequential: to add layers in sequence, to initialize ann model i.e. initially random weight will be assigned
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('/content/Churn_Modelling.csv')
df # Exited is a target column. Binary Classification problem
# given the customer details, bank want to predict whether customer will leave the bank or not.

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
# Not important columns
df.drop(columns=['RowNumber','CustomerId','Surname'], inplace=True)

In [5]:
df.shape

(10000, 11)

# Data Transformation

In [6]:
df.isna().sum()

,0
CreditScore,0
Geography,0
Gender,0
Age,0
Tenure,0
Balance,0
NumOfProducts,0
HasCrCard,0
IsActiveMember,0
EstimatedSalary,0


In [7]:
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [8]:
df.drop(columns = ['Exited']) # no permanent deletion of Exited column

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,Female,42,2,0.00,1,1,1,101348.88
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58
2,502,France,Female,42,8,159660.80,3,1,0,113931.57
3,699,France,Female,39,1,0.00,2,0,0,93826.63
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10
...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77
9997,709,France,Female,36,7,0.00,1,0,1,42085.58
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52


In [9]:
df = pd.get_dummies(data=df, columns=['Geography','Gender'],dtype = int)  # dtype = int to make encoded values (0,1) permanent
df

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,1,0,0,1,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,0,1,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,1,0,0,1,0
3,699,39,1,0.00,2,0,0,93826.63,0,1,0,0,1,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,0,1,0,0,0,1
9996,516,35,10,57369.61,1,1,1,101699.77,0,1,0,0,0,1
9997,709,36,7,0.00,1,0,1,42085.58,1,1,0,0,1,0
9998,772,42,3,75075.31,2,1,0,92888.52,1,0,1,0,0,1


# Model Building

In [10]:
df

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,1,0,0,1,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,0,1,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,1,0,0,1,0
3,699,39,1,0.00,2,0,0,93826.63,0,1,0,0,1,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,0,1,0,0,0,1
9996,516,35,10,57369.61,1,1,1,101699.77,0,1,0,0,0,1
9997,709,36,7,0.00,1,0,1,42085.58,1,1,0,0,1,0
9998,772,42,3,75075.31,2,1,0,92888.52,1,0,1,0,0,1


In [11]:
sc = StandardScaler() # don't perform standardization on target column
x = df.drop(columns=['Exited'])# exclude Exited column
y = df['Exited']

In [12]:
x = sc.fit_transform(x)
x

array([[-0.32622142,  0.29351742, -1.04175968, ..., -0.57380915,
         1.09598752, -1.09598752],
       [-0.44003595,  0.19816383, -1.38753759, ...,  1.74273971,
         1.09598752, -1.09598752],
       [-1.53679418,  0.29351742,  1.03290776, ..., -0.57380915,
         1.09598752, -1.09598752],
       ...,
       [ 0.60498839, -0.27860412,  0.68712986, ..., -0.57380915,
         1.09598752, -1.09598752],
       [ 1.25683526,  0.29351742, -0.69598177, ..., -0.57380915,
        -0.91241915,  0.91241915],
       [ 1.46377078, -1.04143285, -0.35020386, ..., -0.57380915,
         1.09598752, -1.09598752]])

In [13]:
xtrain,xtest,ytrain,ytest = train_test_split(x,y,test_size=0.20, random_state=1)

In [14]:
xtrain

array([[-0.23310044, -0.94607926, -0.69598177, ..., -0.57380915,
        -0.91241915,  0.91241915],
       [-0.25379399, -0.94607926, -0.35020386, ..., -0.57380915,
         1.09598752, -1.09598752],
       [-0.39864885,  0.77028538,  0.34135195, ..., -0.57380915,
         1.09598752, -1.09598752],
       ...,
       [ 0.22215769,  0.5795782 ,  1.37868567, ..., -0.57380915,
         1.09598752, -1.09598752],
       [ 0.12903671,  0.00745665,  1.03290776, ..., -0.57380915,
         1.09598752, -1.09598752],
       [ 1.16371428,  0.29351742,  0.34135195, ..., -0.57380915,
        -0.91241915,  0.91241915]])

In [15]:
# create instance of Sequential class or initialize the model
model = Sequential()
# Sequential is a class in Keras that allows you to build ANN model layer-by-layer.
# create neurons in input, hidden and output layers and assign random weights to input
# In I/P layer 13 input values will be there as there are 13 independent features

#Add hidden layer, randomly 10 neurons are added
model.add(Dense(units=10,activation='relu'))# as target is binary use relu activation
# Layers are added sequentially using the .add() method.
# Each layer receives input from the previous layer and passes output to the next one.

#Add output layer
model.add(Dense(units=1, activation='sigmoid'))# for binary classification use sigmoid activation in output layer

#Establish the connection between the layers
model.compile(optimizer = 'adadelta',loss='binary_crossentropy',metrics=['accuracy'])
# adadelta is gradient descent algorithm for weight updation which we have selected randomly. We will hypertune it afterwards
# binary_crossentropy: also called log_loss, finds erros, range is 0 to 1

#Fit the data, perform forward and back propagation
model.fit(xtrain,ytrain, epochs=100)
# epochs=100 means that the model will go through the entire xtrain and ytrain dataset 100 times during training.
# epoch: backward and forward propagation will be done for 100 times
# The number of batches per epoch = Total number of samples / Batch size (default is 32)
# 80% training data = 8000 samples. So Batches=8000/32 = 250
# Each batch is processed in sequence during each epoch,
# and after all 250 batches are processed, one epoch is complete.

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4153 - loss: 0.8463
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4113 - loss: 0.8396
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4078 - loss: 0.8493
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4151 - loss: 0.8387
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4158 - loss: 0.8345
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4005 - loss: 0.8504
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4062 - loss: 0.8364
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4178 - loss: 0.8385
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4253 - loss: 0.8261
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4226 - loss: 0.8314
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4262 - loss: 0.8247
Epoch 12/100
250/250 ━━━━━━━━━━━━━━━━━━━━

In [16]:
# accuracy is not good and error is also high as hyperparameter tuning is not done
xtest

array([[-1.04014895,  0.77028538, -1.04175968, ..., -0.57380915,
        -0.91241915,  0.91241915],
       [ 0.3049319 , -0.4693113 , -0.69598177, ..., -0.57380915,
        -0.91241915,  0.91241915],
       [-1.23673768,  0.29351742, -1.04175968, ..., -0.57380915,
         1.09598752, -1.09598752],
       ...,
       [-0.86425376, -0.4693113 ,  1.72446358, ...,  1.74273971,
        -0.91241915,  0.91241915],
       [-0.30552787, -0.85072567, -1.04175968, ..., -0.57380915,
         1.09598752, -1.09598752],
       [ 0.0462625 ,  1.24705333,  1.37868567, ..., -0.57380915,
        -0.91241915,  0.91241915]])

In [17]:
ypred = model.predict(xtest)
ypred # in dataset target column values are 0 or 1, but here output is a continuous number
# because in output layer activation function is sigmoid. It gives you o/p in terms of probability
# In sigmoid if probability is > 0.5 it gives you 1 and if probability < 0 it should give 0

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


array([[0.58954424],
       [0.6683034 ],
       [0.50166833],
       ...,
       [0.4238216 ],
       [0.3591241 ],
       [0.47521642]], dtype=float32)

In [18]:
ytest

,Exited
9953,0
3850,0
4962,0
3886,0
5437,0
...,...
3919,0
162,0
7903,0
2242,0


In [19]:
print(classification_report(ytest,ypred))

ValueError: Classification metrics can't handle a mix of binary and continuous targets

In [20]:
ypred > 0.5 # not it will give me 1 or 0 OR true or false answers

array([[ True],
       [ True],
       [ True],
       ...,
       [False],
       [False],
       [False]])

In [21]:
ypred = ypred > 0.5
ypred

array([[ True],
       [ True],
       [ True],
       ...,
       [False],
       [False],
       [False]])

In [22]:
ytest

,Exited
9953,0
3850,0
4962,0
3886,0
5437,0
...,...
3919,0
162,0
7903,0
2242,0


In [23]:
print(classification_report(ytest,ypred))

              precision    recall  f1-score   support

           0       0.81      0.57      0.67      1585
           1       0.23      0.49      0.32       415

    accuracy                           0.56      2000
   macro avg       0.52      0.53      0.49      2000
weighted avg       0.69      0.56      0.60      2000



# Hyperparameter Tuning

In [24]:
!pip install -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 2.8 MB/s eta 0:00:00


# **Tuning the ANN Model**
- Selecting the best optimizer




In [3]:
import keras_tuner as kt

ModuleNotFoundError: No module named 'keras_tuner'

In [26]:
# tuning for optimizer(adam,rmsprop or sgd:stochastic gradient descent) only
def optimizer_selection(hp):
    # hp (HyperParameters) is an object used to define search spaces for the hyperparameters you want to tune.
    # This allows the Keras Tuner to explore different values for the hyperparameters
    # and find the best ones during the search.

    #create instance of sequential class
    model = Sequential()
    #Add hidden layer
    model.add(Dense(units=10, activation='relu'))
    #Add output layer
    model.add(Dense(units=1, activation='sigmoid'))
    #Optimizer selection
    optim = hp.Choice('optimizer', values = ['sgd','adam','rmsprop']) #value of optimizer is categorical so use choice()function else use hp.Int()
    model.compile(optimizer=optim, loss = 'binary_crossentropy', metrics = ['accuracy'])
    return model

In [27]:
# similar to GridSearchCV() we will use here RandomSearch()
tuner = kt.RandomSearch(
    optimizer_selection,# model name-here Sequential() class is our model
    objective='val_accuracy', # increase accuracy of test data i.e. validation test accuracy or minimize 'val_loss'
    max_trials=3 # for each optimizer take 3 trials: for sgd-3 trials, for adam-3 trials, for rmsprop-3 trials
    # can select 3 to 10 or 20 for larger problems
)

In [28]:
tuner.search(xtrain,ytrain, epochs = 3, validation_data = (xtest,ytest))
# tuner.search() is equivalent to grid.fit()
# When we call tuner.search(), we are not directly calling the hyper() function.
# Instead, Keras Tuner automatically calls hyper()
# and passes the hp object to it during the search.
# We don’t manually pass hp as an argument because Keras Tuner handles it behind the scenes.

# epochs should be same as max_trials so 3 is selected
# validation_data: pass test data

Trial 3 Complete [00h 00m 04s]
val_accuracy: 0.8134999871253967

Best val_accuracy So Far: 0.8330000042915344
Total elapsed time: 00h 00m 12s


In [29]:
tuner.get_best_hyperparameters()[0].values # as per performance all optimizers are stored in an array
# we want high accuracy optimizer present at index 0 location

{'optimizer': 'rmsprop'}

In [30]:
model = tuner.get_best_models(num_models=1)[0] # get best model: rmsprop
model.fit(xtrain,ytrain, epochs = 100, validation_data = (xtest,ytest))

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8294 - loss: 0.4037 - val_accuracy: 0.8370 - val_loss: 0.3960
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8378 - loss: 0.3913 - val_accuracy: 0.8395 - val_loss: 0.3866
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8312 - loss: 0.3921 - val_accuracy: 0.8415 - val_loss: 0.3795
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8505 - loss: 0.3686 - val_accuracy: 0.8465 - val_loss: 0.3738
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8462 - loss: 0.3697 - val_accuracy: 0.8450 - val_loss: 0.3683
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8568 - loss: 0.3511 - val_accuracy: 0.8530 - val_loss: 0.3637
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8539 - loss: 0.3538 - val_accuracy: 0.8510 - val_loss: 0.3604
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8463 - loss: 0.3689 - val_accu

In [31]:
# In above output we have only hypertuned optimizer and
# we got generalized model having improved accuracy- previously it was 76%
# now we got training accuracy=86.47% and testing accuracy=86.40%
model.evaluate(xtrain,ytrain) # same as score() function


250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8665 - loss: 0.3308


[0.33096081018447876, 0.8634999990463257]

In [32]:
model.evaluate(xtest,ytest)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8575 - loss: 0.3383


[0.33323630690574646, 0.8619999885559082]

# Regression

In [8]:
df = pd.read_csv('/content/Price.csv')
df

,price,feature1,feature2
0,461.527929,999.787558,999.766096
1,548.130012,998.861615,1001.042403
2,410.297162,1000.070267,998.844015
3,540.382220,999.952251,1000.440940
4,546.024553,1000.446011,1000.338531
...,...,...,...
995,476.526078,1000.018988,999.672732
996,457.313186,998.855379,1000.020026
997,456.720993,1001.451646,998.847605
998,403.315576,1000.771023,998.562851


In [9]:
x = df.iloc[:,1:]
y = df['price']

In [10]:
sc = StandardScaler()
x = sc.fit_transform(x)
x

array([[-0.23277509, -0.22551031],
       [-1.18389307,  1.12100979],
       [ 0.05762098, -1.19831827],
       ...,
       [ 1.47655818, -1.19452982],
       [ 0.77742978, -1.49494959],
       [-0.80318737,  1.55251428]])

In [11]:
xtrain,xtest,ytrain,ytest = train_test_split(x,y,test_size=0.2, random_state=1)

In [12]:
ann = Sequential()

ann.add(Dense(units=30, activation='relu'))# hidden layer1
ann.add(Dense(units=20, activation='relu'))# hidden layer2

ann.add(Dense(units=1))# output layer.
#In Regression task no activation function is required in output layer as we don't have to convert price to 0 or 1

ann.compile(optimizer='adam',loss = 'mse')

ann.fit(xtrain,ytrain, epochs = 100, validation_data = (xtest,ytest))

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 256630.9062 - val_loss: 262447.7188
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 254459.5781 - val_loss: 261697.6562
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 247372.5312 - val_loss: 260517.6875
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 251385.0938 - val_loss: 258649.3125
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 250179.0156 - val_loss: 255895.5469
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 245108.9531 - val_loss: 252049.4062
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 242336.2500 - val_loss: 246875.7656
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 237858.9219 - val_loss: 240216.1562
Epoch 9/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 232794.7656 - val_loss: 231784.2344
Epoch 10/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 219611.6562 - val_loss: 221559.2969
Epoch 11/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/

In [21]:
yp = ann.predict(xtest)
yp

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


array([[480.23456],
       [529.1278 ],
       [499.31482],
       [538.4315 ],
       [646.9514 ],
       [344.4637 ],
       [341.34607],
       [373.39517],
       [710.22876],
       [504.92322],
       [482.75314],
       [511.03958],
       [499.69785],
       [366.22   ],
       [550.458  ],
       [488.94028],
       [577.1356 ],
       [551.3544 ],
       [487.2112 ],
       [504.13077],
       [470.53885],
       [602.8876 ],
       [445.228  ],
       [572.27734],
       [427.68954],
       [524.7731 ],
       [382.01523],
       [497.4301 ],
       [545.9923 ],
       [339.69974],
       [443.78088],
       [640.70215],
       [561.5419 ],
       [539.98206],
       [608.63086],
       [551.37415],
       [406.62567],
       [575.0232 ],
       [558.1842 ],
       [523.50305],
       [425.6635 ],
       [718.65625],
       [527.614  ],
       [535.18896],
       [518.71484],
       [455.59122],
       [528.9337 ],
       [521.6958 ],
       [586.65344],
       [471.43484],


In [26]:
new_input = np.array([[999.787558,999.766096]])

In [29]:
# The model must receive the input in the same preprocessing/scale as your training data.
# If you scaled xtrain with StandardScaler or MinMaxScaler, you must also scale the new input using the same scaler before predicting.
new_input_scaled = sc.transform([[999.787558,999.766096]])
predicted_price = ann.predict(new_input_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


In [30]:
print("Predicted Price:", predicted_price)

Predicted Price: [[471.8128]]


In [ ]:
from sklearn.metrics import r2_score # accuracy metric for regression task is r squared

In [ ]:
r2_score(ytest,yp)

0.976023762078309